# Stage 2 — Earth Network Explorer

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 2 — Earth interactive network exploration |
| **Previous stage** | Stage 1 — Source data QA (`00_earth_source_data_qa.ipynb`) |
| **Next stage** | Stage 3 — Mars network exploration (`notebooks/mars/00_mars_network_explorer.ipynb`) |
| **Purpose** | Interactive per-basin threshold and pruning preview. Set `BASIN` and `THRESHOLD_KM2` below and re-run to explore how parameters affect network structure, drainage density, and Strahler order distribution. |
| **Inputs** | `data/cropped_DEMs/<basin>.tif`; `channel_heads.regimes` for regime reference lines |
| **Outputs** | Network visualisation, DD metrics, Strahler distribution. Nothing written to disk. |
| **Decision gate** | Informational — use to develop intuition before Stage 4 regime calibration. |

## 0. Configuration — change these and re-run

In [1]:
# Basin to explore — one of: calnalpine, daqing, finisterre, humboldt, inyo,
# kammanasie, luliang, panamint, sakhalin, sierramadre, sierranevadaspain,
# taiwan, toano, troodos, tsugaru, vallefertil, yoro
BASIN = "inyo"

# Stream area threshold in km² (controls network density)
THRESHOLD_KM2 = 0.1

# Regime presets shown as reference lines on DD plot
SHOW_REGIME_LINES = True

## 1. Imports

In [2]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from channel_heads.io.paths import EXAMPLE_DEMS, RESULTS_DIR
from channel_heads.basin_config import get_basin_config, LOCAL_TO_PAPER_BASIN
from channel_heads.dd_calibration import collect_basin_metrics_for_dem
from channel_heads.regimes import REGIMES

basin_name = BASIN.lower()
dem_path = EXAMPLE_DEMS.get(basin_name)
if dem_path is None or not dem_path.exists():
    raise FileNotFoundError(f"DEM not found for basin '{basin_name}'. Check EXAMPLE_DEMS.")

cfg = get_basin_config(basin_name)
lat   = cfg["lat"]
z_th  = cfg["z_th"]
paper = LOCAL_TO_PAPER_BASIN.get(basin_name, basin_name)
print(f"Basin  : {basin_name} ({paper})")
print(f"DEM    : {dem_path}")
print(f"Lat    : {lat}°  z_th : {z_th} m")
print(f"Threshold: {THRESHOLD_KM2} km²")

Basin  : inyo (inyo)
DEM    : /Users/guypi/Projects/channel-heads/data/cropped_DEMs/Inyo_strm_crop.tif
Lat    : 36.71°  z_th : 1200 m
Threshold: 0.1 km²


## 2. Build stream network at chosen threshold

In [3]:
results = collect_basin_metrics_for_dem(
    dem_path=dem_path,
    basin_name=basin_name,
    thresholds_km2=[THRESHOLD_KM2],
    z_th=z_th,
    lat_deg=lat,
)
if not results:
    raise RuntimeError(f"No stream network found for '{basin_name}' at T={THRESHOLD_KM2} km².")
metrics = max(results, key=lambda m: m.n_stream_nodes)

print(f"Drainage density : {metrics.dd_true_km_km2:.3f} km/km²")
print(f"Total stream length : {metrics.stream_length_km:.1f} km")
print(f"Basin area : {metrics.basin_area_km2:.1f} km²")
print(f"Stream nodes : {metrics.n_stream_nodes}")

Drainage density : 1.873 km/km²
Total stream length : 43.8 km
Basin area : 23.4 km²
Stream nodes : 460


## 3. Strahler order distribution

In [4]:
if hasattr(metrics, 'strahler_distribution') and metrics.strahler_distribution:
    dist = metrics.strahler_distribution
    orders = sorted(dist.keys())
    counts = [dist[o] for o in orders]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(orders, counts, color='steelblue', edgecolor='white')
    ax.set_xlabel('Strahler order')
    ax.set_ylabel('Node count')
    ax.set_title(f'{paper} — Strahler distribution (T={THRESHOLD_KM2} km²)')
    ax.set_xticks(orders)
    fig.tight_layout()
    plt.show()
else:
    print("Strahler distribution not available in metrics object.")

Strahler distribution not available in metrics object.


## 4. Threshold sweep — drainage density vs T

In [5]:
from collections import defaultdict

SWEEP_THRESHOLDS = [0.02, 0.05, 0.10, 0.15, 0.25, 0.40, 0.60, 1.00, 2.00]

all_sweep = collect_basin_metrics_for_dem(
    dem_path=dem_path,
    basin_name=basin_name,
    thresholds_km2=SWEEP_THRESHOLDS,
    z_th=z_th,
    lat_deg=lat,
)
by_thr: dict = defaultdict(list)
for _m in all_sweep:
    by_thr[_m.threshold_km2].append(_m)

sweep_rows = []
for t in SWEEP_THRESHOLDS:
    ms = by_thr.get(t, [])
    if not ms:
        continue
    m = max(ms, key=lambda x: x.n_stream_nodes)
    sweep_rows.append({
        "threshold_km2": t,
        "dd": m.dd_true_km_km2,
        "length_km": m.stream_length_km,
        "nodes": m.n_stream_nodes,
    })
    print(f"T={t:5.2f} km²  DD={m.dd_true_km_km2:.3f} km/km²  nodes={m.n_stream_nodes}")

T= 0.02 km²  DD=7.602 km/km²  nodes=1820
T= 0.05 km²  DD=3.343 km/km²  nodes=813
T= 0.10 km²  DD=1.873 km/km²  nodes=460
T= 0.15 km²  DD=1.509 km/km²  nodes=372
T= 0.25 km²  DD=1.227 km/km²  nodes=304
T= 0.40 km²  DD=0.958 km/km²  nodes=239
T= 0.60 km²  DD=0.770 km/km²  nodes=192
T= 1.00 km²  DD=0.700 km/km²  nodes=156
T= 2.00 km²  DD=0.559 km/km²  nodes=126


In [6]:
import pandas as pd

sweep_df = pd.DataFrame(sweep_rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# DD vs threshold
ax = axes[0]
ax.plot(sweep_df['threshold_km2'], sweep_df['dd'], 'o-', color='steelblue')
ax.set_xlabel('Threshold (km²)')
ax.set_ylabel('Drainage density (km/km²)')
ax.set_xscale('log')
ax.set_title(f'{paper} — DD vs threshold')

if SHOW_REGIME_LINES:
    colors = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}
    for reg_name, regime in REGIMES.items():
        ax.axvline(regime.threshold_km2, color=colors[reg_name],
                   linestyle='--', linewidth=1.2, label=reg_name)
    ax.legend(fontsize=8)

ax.axvline(THRESHOLD_KM2, color='black', linestyle='-', linewidth=1.5,
           label=f'Current ({THRESHOLD_KM2} km²)')
ax.legend(fontsize=8)

# Stream length vs threshold
ax2 = axes[1]
ax2.plot(sweep_df['threshold_km2'], sweep_df['length_km'], 'o-', color='darkorange')
ax2.set_xlabel('Threshold (km²)')
ax2.set_ylabel('Total stream length (km)')
ax2.set_xscale('log')
ax2.set_title(f'{paper} — Stream length vs threshold')
ax2.axvline(THRESHOLD_KM2, color='black', linestyle='-', linewidth=1.5)

fig.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_50341/343204894.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Multi-basin DD comparison at regime thresholds

Run for all 17 basins at each regime threshold to see the DD spread.  
Set `RUN_MULTI_BASIN = True` — this takes ~1–3 min.

In [7]:
RUN_MULTI_BASIN = False  # set True to run for all basins

if RUN_MULTI_BASIN:
    mb_rows = []
    for bname, bpath in sorted(EXAMPLE_DEMS.items()):
        try:
            bcfg = get_basin_config(bname)
        except KeyError:
            continue
        for reg_name, regime in REGIMES.items():
            try:
                res = collect_basin_metrics_for_dem(
                    dem_path=bpath,
                    basin_name=bname,
                    thresholds_km2=[regime.threshold_km2],
                    z_th=bcfg['z_th'],
                    lat_deg=bcfg['lat'],
                )
                if not res:
                    continue
                m = max(res, key=lambda x: x.n_stream_nodes)
                mb_rows.append({
                    'basin': bname,
                    'regime': reg_name,
                    'dd': m.dd_true_km_km2,
                })
            except Exception as exc:
                print(f"  {bname}/{reg_name}: {exc}")

    mb_df = pd.DataFrame(mb_rows)
    fig, ax = plt.subplots(figsize=(12, 4))
    colors = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}
    basins_sorted = mb_df[mb_df['regime'] == 'regA'].sort_values('dd')['basin'].tolist()
    x = range(len(basins_sorted))
    width = 0.28
    for i, (reg_name, _) in enumerate(REGIMES.items()):
        sub = mb_df[mb_df['regime'] == reg_name].set_index('basin').reindex(basins_sorted)
        ax.bar([xi + i * width for xi in x], sub['dd'],
               width=width, label=reg_name, color=colors[reg_name], alpha=0.8)
    ax.set_xticks([xi + width for xi in x])
    ax.set_xticklabels(basins_sorted, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Drainage density (km/km²)')
    ax.set_title('Earth DD by basin and regime')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Set RUN_MULTI_BASIN = True to run DD for all basins at all regime thresholds.")

Set RUN_MULTI_BASIN = True to run DD for all basins at all regime thresholds.
